# Technical PDF RAG Assistant with Hybrid Retrieval and

Load functions from src folder.

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

## Load Files:

Testing parse_pdf_with_metadata function using langchain_community PyMuPDFLoader.

In [7]:
from src.ResearchRAG.ingestion.loaders import parse_pdf_folder

folder_path = 'data/raw/'
files = parse_pdf_folder(folder_path)
print("Total pages: ", len(files))

Total pages:  155


In [8]:
print("First page content:\n", files[0].page_content)

First page content:
 SPECIAL SECTION ON ARTIFICIAL INTELLIGENCE (AI)-EMPOWERED
INTELLIGENT TRANSPORTATION SYSTEMS
Received November 10, 2019, accepted November 21, 2019, date of publication December 6, 2019,
date of current version December 23, 2019.
Digital Object Identifier 10.1109/ACCESS.2019.2958068
Anomaly Detection Based on Spatio-Temporal
and Sparse Features of Network Traffic in VANETs
LAISEN NIE
1,2, YIXUAN WU
1, HUIZHI WANG
1, AND YONGKANG LI
1
1School of Electronics and Information, Northwestern Polytechnical University, Xi’an 710072, China
2Qingdao Research Institute, Northwestern Polytechnical University, Qingdao, China
Corresponding author: Laisen Nie (nielaisen@nwpu.edu.cn)
This work was supported in part by the National Natural Science Foundation of China under Grant 61701406 and Grant 61701413, in part
by the Applied Basic Research Programs of Qingdao City under Grant 18-2-2-36-jch, in part by the China Postdoctoral Science
Foundation under Grant BX20180261, in part by

In [9]:
print("First page metadata:\n", files[0].metadata)

First page metadata:
 {'producer': 'pdfTeX-1.40.13; modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-12-14T12:05:56+05:30', 'source': 'C:\\Users\\pstfc\\PycharmProjects\\GenAI\\data\\raw\\Anomaly_Detection_Based_on_Spatio-Temporal_and_Sparse_Features_of_Network_Traffic_in_VANETs.pdf', 'file_path': 'C:\\Users\\pstfc\\PycharmProjects\\GenAI\\data\\raw\\Anomaly_Detection_Based_on_Spatio-Temporal_and_Sparse_Features_of_Network_Traffic_in_VANETs.pdf', 'total_pages': 11, 'format': 'PDF 1.6', 'title': 'Anomaly Detection Based on Spatio-Temporal and Sparse Features of Network Traffic in VANETs', 'author': 'Laisen Nie', 'subject': 'IEEE Access;2019;7; ;10.1109/ACCESS.2019.2958068', 'keywords': '', 'moddate': '2025-07-30T11:34:18-04:00', 'trapped': '', 'modDate': "D:20250730113418-04'00'", 'creationDate': "D:20191214120556+05'30'", 'page': 0}


## Data Cleaning:

In [10]:
from src.ResearchRAG.preprocessing import clean_text
for i,text in enumerate(files):
    text.page_content = clean_text(text.page_content)

In [11]:
print("First page cleaned content:\n", files[0].page_content)

First page cleaned content:
 SPECIAL SECTION ON ARTIFICIAL INTELLIGENCE (AI)-EMPOWERED
INTELLIGENT TRANSPORTATION SYSTEMS
Received November 10, 2019, accepted November 21, 2019, date of publication December 6, 2019,
date of current version December 23, 2019.
Digital Object Identifier 10.1109/ACCESS.2019.2958068
Anomaly Detection Based on Spatio-Temporal
and Sparse Features of Network Traffic in VANETs
LAISEN NIE
1,2, YIXUAN WU
1, HUIZHI WANG
1, AND YONGKANG LI
1
1School of Electronics and Information, Northwestern Polytechnical University, Xi’an 710072, China
2Qingdao Research Institute, Northwestern Polytechnical University, Qingdao, China
Corresponding author: Laisen Nie (nielaisen@nwpu.edu.cn)
This work was supported in part by the National Natural Science Foundation of China under Grant 61701406 and Grant 61701413, in part
by the Applied Basic Research Programs of Qingdao City under Grant 18-2-2-36-jch, in part by the China Postdoctoral Science
Foundation under Grant BX20180261, in

## Chunking Data:

In [12]:
from src.ResearchRAG.ingestion.chunking import split_text

chunks = split_text(files)
print(chunks)

[Document(metadata={'producer': 'pdfTeX-1.40.13; modified using iText® 7.1.1 ©2000-2018 iText Group NV (AGPL-version)', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-12-14T12:05:56+05:30', 'source': 'C:\\Users\\pstfc\\PycharmProjects\\GenAI\\data\\raw\\Anomaly_Detection_Based_on_Spatio-Temporal_and_Sparse_Features_of_Network_Traffic_in_VANETs.pdf', 'file_path': 'C:\\Users\\pstfc\\PycharmProjects\\GenAI\\data\\raw\\Anomaly_Detection_Based_on_Spatio-Temporal_and_Sparse_Features_of_Network_Traffic_in_VANETs.pdf', 'total_pages': 11, 'format': 'PDF 1.6', 'title': 'Anomaly Detection Based on Spatio-Temporal and Sparse Features of Network Traffic in VANETs', 'author': 'Laisen Nie', 'subject': 'IEEE Access;2019;7; ;10.1109/ACCESS.2019.2958068', 'keywords': '', 'moddate': '2025-07-30T11:34:18-04:00', 'trapped': '', 'modDate': "D:20250730113418-04'00'", 'creationDate': "D:20191214120556+05'30'", 'page': 0}, page_content='SPECIAL SECTION ON ARTIFICIAL INTELLIGENCE (AI)-EMPOWERED

## Embedding:

In [17]:
from src.ResearchRAG.embedding.embeddings import build_embedding_model

minilm_embedding_model = build_embedding_model("miniLM")
coher_embedding_model = build_embedding_model("cohere_v3")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11014.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Indexing:

In [15]:

from dotenv import load_dotenv
import os
from src.ResearchRAG.config import ROOT_DIR

# If your notebook is inside the project root, this is enough:
load_dotenv(ROOT_DIR / ".env")

print(os.getenv("COHERE_API_KEY"))

LPFkWJTN3lmLZzSWdBlkYOjOIcxQyLemtPc54xsr


In [18]:
from src.ResearchRAG.embedding.vectorstore import build_database, save_faiss_index
vectorstore = build_database(chunks, minilm_embedding_model)
save_faiss_index(vectorstore, "faiss_minilm")

In [19]:
from src.ResearchRAG.embedding.vectorstore import build_database, save_faiss_index
vectorstore = build_database(chunks, coher_embedding_model)
save_faiss_index(vectorstore, "cohere_v3")

## Load Index and Query

In [20]:
from src.ResearchRAG.embedding.vectorstore import load_faiss_index
embedding_model = build_embedding_model("miniLM")
vectorstore = load_faiss_index("faiss_minilm", embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7918.57it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
